In [2]:
import requests
from bs4 import BeautifulSoup
import os
import time
from datetime import datetime

In [ ]:

URL = "https://en.wikipedia.org/wiki/Lionel_Messi"

headers = {
    "User-Agent": "Aula-CPA"
}

# Pastas
output_dir = "data"
html_dir = os.path.join(output_dir, "html")

os.makedirs(output_dir, exist_ok=True)

# Dicionário para armazenar URL e timestamp de cada arquivo baixado
metadata = {}

# Limpar html antigo
import shutil
if os.path.exists(html_dir):
    shutil.rmtree(html_dir)

os.makedirs(html_dir)

# Baixar HTML principal
response = requests.get(URL, headers=headers)
response_soup = BeautifulSoup(response.text, "html.parser")

main_html_path = os.path.join(html_dir, "main.html")

with open(main_html_path, "w", encoding="utf-8") as f:
    f.write(response_soup.prettify())

metadata["main.html"] = {
    "url": URL,
    "timestamp": datetime.now()
}

print("HTML principal salvo")

# Extrair links A PARTIR do HTML salvo
with open(main_html_path, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

links = set()

for a in soup.find_all("a", href=True):
    href = a["href"]

    if href.startswith("/wiki/") and ":" not in href:
        full_link = "https://en.wikipedia.org" + href
        links.add(full_link)

links = list(links)

print(f"{len(links)} links encontrados")

# Baixar páginas dos links
for i, link in enumerate(links): # se nao quiser baixar os 2000 links, pode usar (links[:10]) por exemplo
    try:
        r = requests.get(link, headers=headers)
        r_soup = BeautifulSoup(r.text, "html.parser")

        file_name = f"page_{i}.html"
        file_path = os.path.join(html_dir, f"page_{i}.html")

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(r_soup.prettify())

        metadata[file_name] = {
            "url": link,
            "timestamp": datetime.now()
        }

        time.sleep(1)

    except:
        continue

print("Download concluído")

HTML principal salvo
2051 links encontrados
Download concluído


In [ ]:
import pandas as pd

# Reconstrói o caminho da pasta
html_dir = os.path.join(output_dir, "html")

# Sempre reinicia (evita duplicação)
data_pages = []
data_images = []

# Coleta imagens apenas do artigo principal
with open(os.path.join(html_dir, "main.html"), "r", encoding="utf-8") as f:
    main_soup = BeautifulSoup(f, "html.parser")

images = [
    img["src"] for img in main_soup.find_all("img")
    if img.get("src")
]

for img in images:
    data_images.append({
        "image_url": "https:" + img if img.startswith("//") else img
    })

# Coleta e ordena os arquivos HTMLs 
files = os.listdir(html_dir)
files = sorted(files, key=lambda f: (0 if f == "main.html" else 1, f))

# Itera sobre todos os HTMLs salvos para extrair os dados de cada página
for file in files:
    path = os.path.join(html_dir, file)

    if not os.path.isfile(path):
        continue

    with open(path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    # Título
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else None

    # Conteúdo principal
    content = soup.find("div", {"id": "mw-content-text"})
    first_paragraph = None

    if content:
        paragraphs = content.find_all("p", recursive=True)

        for p in paragraphs:
            text = p.get_text(strip=True)
            if text:
                first_paragraph = text
                break

    # Links internos
    internal_links = [
        a["href"] for a in soup.find_all("a", href=True)
        if a["href"].startswith("/wiki/")
    ]

    link_principal = metadata[file]["url"]
    timestamp = metadata[file]["timestamp"]

    # Salva uma vez só
    data_pages.append({
        "title": title,
        "first_paragraph": first_paragraph,
        "link_principal": link_principal,
        "internal_links": ["https://en.wikipedia.org" + link for link in internal_links],
        "timestamp": timestamp
    })

In [ ]:
# Converte os dados coletados em DataFrames e exporta para CSV

df_pages = pd.DataFrame(data_pages)
df_images = pd.DataFrame(data_images)

df_pages.to_csv("data/pages.csv",sep=";", index=False)
df_images.to_csv("data/images.csv", sep=";", index=False)

In [13]:
df_pages.tail(20)

,title,first_paragraph,link_principal,internal_links,timestamp
0,Lionel Messi,"Lionel Andrés""Leo""Messi[note 1](born 24 June 1...",https://en.wikipedia.org/wiki/Lionel_Messi,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 19:55:20.541588
1,2016–17 UEFA Champions League,The2016–17 UEFA Champions Leaguewas the 62nd s...,https://en.wikipedia.org/wiki/2016%E2%80%9317_...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 19:55:27.074570
2,Ballon d'Or,TheBallon d'Or(French pronunciation:[balɔ̃dɔʁ]...,https://en.wikipedia.org/wiki/Ballon_d%27Or,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 19:55:29.244813
3,Marcel Hirscher,Marcel Hirscher(born 2 March 1989)[1]is an Aus...,https://en.wikipedia.org/wiki/Marcel_Hirscher,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 19:55:31.134859
4,Libertad Lamarque,Libertad Lamarque Bouza(Spanish pronunciation:...,https://en.wikipedia.org/wiki/Libertad_Lamarque,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 19:55:32.594152
5,Michel Platini,Michel François Platini(French pronunciation:[...,https://en.wikipedia.org/wiki/Michel_Platini,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 19:55:34.873609
6,Manuel Badenes,Manuel Badenes Calduch(31 October 1928 – 26 No...,https://en.wikipedia.org/wiki/Manuel_Badenes,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 19:55:36.204708
7,2005 FIFA Club World Championship,The2005 FIFA Club World Championship(officiall...,https://en.wikipedia.org/wiki/2005_FIFA_Club_W...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 19:55:38.360324
8,1978 Ballon d'Or,"The1978 Ballon d'Or, given to the bestfootball...",https://en.wikipedia.org/wiki/1978_Ballon_d%27Or,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 19:55:40.462867
9,The Best FIFA Football Awards 2018,The Best FIFA Football Awards2018were held on ...,https://en.wikipedia.org/wiki/The_Best_FIFA_Fo...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 19:55:42.096519
